# Practice: attention, by hand and in PyTorch
*AI from scratch, chapter 7* · lesson: https://ihor.sh/ai/07-attention.html

Write scaled dot-product attention yourself and check it against PyTorch's, add the causal mask, split it into
heads, give positions a signal, and run a real transformer layer. Every large language model is stacks of this.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Use masked_fill to replace future positions with negative infinity so their softmax weight becomes 0.', 'scores = scores.masked_fill(mask, float("-inf"))'], 2: ['w = scores.softmax(-1) turns each row of scores into weights that add up to 1.', 'w = scores.softmax(-1), then return w @ V, w'], 3: ['10,000 dot products of random d-long vectors: (torch.randn(10_000, d) * torch.randn(10_000, d)).sum(1)', 'spread = [(torch.randn(10_000, d) * torch.randn(10_000, d)).sum(1).std().item() for d in (4, 64, 1024)]'], 4: ['In self-attention, query, key, and value are all the same input tensor x passed to mha.', 'Forward pass self-attention: out, w = mha(x, x, x).'], 5: ['Form position indices p = np.arange(L)[:, None] and dimension pairs i = np.arange(d // 2)[None, :].', 'Compute even indices with sin: PE[:, 0::2] = np.sin(p / 10000 ** (2 * i / d)).', 'Compute odd indices with cos: PE[:, 1::2] = np.cos(p / 10000 ** (2 * i / d)).'], 6: ['y = enc(x, mask=mask, is_causal=True)', 'n_params = sum(p.numel() for p in enc.parameters())'], 7: ['y2 = enc(x2, mask=mask, is_causal=True), then compare y2[:, :6] with y[:, :6].', 'unchanged = torch.allclose(enc(x2, mask=mask, is_causal=True)[:, :6], y[:, :6], atol=1e-5)']}
_ANSWERS = {1: 'c2NvcmVzID0gc2NvcmVzLm1hc2tlZF9maWxsKG1hc2ssIGZsb2F0KCItaW5mIikp', 2: 'dyA9IHNjb3Jlcy5zb2Z0bWF4KC0xKQpyZXR1cm4gdyBAIFYsIHc=', 3: 'c3ByZWFkID0gWyh0b3JjaC5yYW5kbigxMF8wMDAsIGQpICogdG9yY2gucmFuZG4oMTBfMDAwLCBkKSkuc3VtKDEpLnN0ZCgpLml0ZW0oKSBmb3IgZCBpbiAoNCwgNjQsIDEwMjQpXQ==', 4: 'b3V0LCB3ID0gbWhhKHgsIHgsIHgp', 5: 'cCA9IG5wLmFyYW5nZShMKVs6LCBOb25lXTsgaSA9IG5wLmFyYW5nZShkIC8vIDIpW05vbmUsIDpdClBFID0gbnAuemVyb3MoKEwsIGQpKTsgUEVbOiwgMDo6Ml0gPSBucC5zaW4ocCAvIDEwMDAwICoqICgyICogaSAvIGQpKTsgUEVbOiwgMTo6Ml0gPSBucC5jb3MocCAvIDEwMDAwICoqICgyICogaSAvIGQpKQpyZXR1cm4gUEU=', 6: 'eSA9IGVuYyh4LCBtYXNrPW1hc2ssIGlzX2NhdXNhbD1UcnVlKQpuX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gZW5jLnBhcmFtZXRlcnMoKSk=', 7: 'dW5jaGFuZ2VkID0gdG9yY2guYWxsY2xvc2UoZW5jKHgyLCBtYXNrPW1hc2ssIGlzX2NhdXNhbD1UcnVlKVs6LCA6Nl0sIHlbOiwgOjZdLCBhdG9sPTFlLTUp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Scaled dot-product attention
`attention(Q, K, V, causal=False)` works on tensors of shape (batch, length, d). The scores $QK^\top/\sqrt{d}$ are
computed for you, and so is `mask`: True above the diagonal, the positions in each token's future.
Two steps are yours. First, when `causal`, set the masked scores to $-\infty$: `scores.masked_fill(mask, value)`
puts `value` wherever `mask` is True, and `float("-inf")` is $-\infty$ in Python. Then take the softmax over the
last dimension and multiply by V, returning (output, weights), as in the lesson's NumPy version.

In [ ]:
def attention(Q, K, V, causal=False):
    """Scaled Dot-Product Attention: Softmax(Q @ K.T / sqrt(d_k)) @ V.
    
    Args:
        Q: Query tensor of shape (..., seq_len, d_k).
        K: Key tensor of shape (..., seq_len, d_k).
        V: Value tensor of shape (..., seq_len, d_v).
        causal: Boolean, if True applies upper-triangular autoregressive mask.
    Returns:
        tuple: (output, weights) where output has shape (..., seq_len, d_v) and weights has shape (..., seq_len, seq_len).
    """
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if causal:
        L = Q.shape[-2]
        mask = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)     # True where a token would see its future
        # YOUR CODE HERE (stuck? hint(1), then answer(1))
        ...
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

Q, K, V = torch.randn(3, 5, 8, dtype=torch.float64).unbind(0)
Q, K, V = Q[None], K[None], V[None]                    # batch of 1, 5 positions, d = 8

In [ ]:
check(2, lambda: torch.allclose(attention(Q, K, V)[0], F.scaled_dot_product_attention(Q, K, V)), 
      "Non-causal attention must match PyTorch's F.scaled_dot_product_attention.")
check(2, lambda: torch.allclose(attention(Q, K, V, causal=True)[0], F.scaled_dot_product_attention(Q, K, V, is_causal=True)), 
      "Causal attention must match PyTorch's F.scaled_dot_product_attention(..., is_causal=True).")
check(2, lambda: torch.allclose(attention(Q, K, V, causal=True)[1].triu(1), torch.zeros(1, 5, 5, dtype=torch.float64)), 
      "Causal attention weights above the main diagonal must be strictly zero.")

## 2. Why divide by √d?
With random vectors, a dot product's spread grows like √d. Compute the standard deviation of the scores for
d = 4, 64 and 1024 **without** the scaling (`q @ k` for 10,000 random pairs), and the largest softmax weight that
typical row gets. Set `spread` (a list). Without scaling, big d makes softmax nearly one-hot and gradients vanish.

In [ ]:
spread = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(spread)

In [ ]:
check(3, lambda: abs(spread[0] - 2) < 0.2 and abs(spread[2] - 32) < 2, 
      "Unscaled dot product standard deviation grows as sqrt(d): ~2 for d=4, ~8 for d=64, ~32 for d=1024.")

## 3. Multi-head attention
Several smaller attentions run side by side, each free to look for something different. PyTorch's
`nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)` does the projections and the split. Run it as
self-attention on `x` (query = key = value = x) and set `out` and `w` (it averages the heads' weights by default).

In [ ]:
mha = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
x = torch.randn(2, 7, 32)                               # 2 sentences, 7 tokens, 32 numbers each
out = w = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(out.shape if out is not None else None, w.shape if w is not None else None)

In [ ]:
check(4, lambda: out.shape == (2, 7, 32) and w.shape == (2, 7, 7), 
      "MultiheadAttention output shape should match input (2, 7, 32), and attention weights shape should be (2, 7, 7).")
check(4, lambda: torch.allclose(w.sum(-1), torch.ones(2, 7)), 
      "Attention weights across the key dimension must sum to 1.0 for each query token.")

## 4. Where am I? Sinusoidal positions
Attention itself ignores order. The original transformer adds a position signal:
$PE[p, 2i] = \sin(p / 10000^{2i/d})$, $PE[p, 2i+1] = \cos(p / 10000^{2i/d})$. Write `positions(L, d)` returning an
(L, d) array, and look at it: slow waves on the right, fast on the left, a unique pattern per row.

In [ ]:
def positions(L, d):
    """Compute sinusoidal positional encodings for sequence length L and dimension d.
    
    Formula:
        PE[pos, 2*i] = sin(pos / 10000^(2*i / d))
        PE[pos, 2*i+1] = cos(pos / 10000^(2*i / d))
        
    Args:
        L: Integer sequence length (number of positions).
        d: Integer embedding dimension (must be even).
    Returns:
        NumPy array of shape (L, d) containing positional encodings.
    """
    # YOUR CODE HERE (stuck? hint(5), then answer(5))
    ...

PE = positions(50, 64)
if PE is not None:
    plt.figure(figsize=(8, 3)); plt.imshow(PE, aspect="auto", cmap="RdBu"); plt.xlabel("dimension"); plt.ylabel("position"); plt.show()

In [ ]:
check(5, lambda: PE.shape == (50, 64) and np.allclose(PE[0, 0::2], 0) and np.allclose(PE[0, 1::2], 1), 
      "PE at position 0 must have sin(0)=0 on even indices and cos(0)=1 on odd indices.")
check(5, lambda: len({tuple(np.round(r, 6)) for r in PE}) == 50, 
      "Each position must receive a unique encoding vector across all 50 positions.")

## 5. A real transformer layer
`nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=128, batch_first=True, norm_first=True)` is the
block from the lesson: attention + residual, MLP + residual, layer norms. `nn.TransformerEncoder` stacks 3 of them,
and `mask` is PyTorch's causal mask for 7 positions (−∞ above the diagonal, as in part 1). Run `x` through the stack
with `enc(x, mask=mask, is_causal=True)` into `y`, and count its parameters into `n_params` (the lesson's
`sum(p.numel() for p in ...parameters())`).

In [ ]:
enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(32, 4, 128, batch_first=True, norm_first=True), num_layers=3).eval()
mask = nn.Transformer.generate_square_subsequent_mask(7)
print(mask)
y = n_params = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print(y.shape if y is not None else None, n_params)

In [ ]:
check(6, lambda: y.shape == x.shape, "the output has the input's shape, (2, 7, 32): one vector per token")
check(6, lambda: n_params == 3 * 12_704, "3 layers × 12,704 parameters (attention 4,224, MLP 8,352, two layer norms 128)")

Now verify the mask works: `x2` is `x` with only the **last** token changed. Run it through `enc` the same way and set
`unchanged`: whether the first six outputs match `y`'s first six (`torch.allclose(a, b, atol=1e-5)`; `[:, :6]` takes
the first six positions).

In [ ]:
x2 = x.clone(); x2[:, -1] += 1.0
unchanged = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print(unchanged)

In [ ]:
check(7, lambda: unchanged is True or unchanged == True, "a causal model can't let token 7 change what tokens 1 to 6 see: unchanged should be True")

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 10, where you train a tiny GPT.